# Conductance of Electrolytic Solutions: Kohlrausch's Law and the $K_a$ of Acetic Acid
*CHEM 3376 — Physical Chemistry Laboratory: Data Analysis & Computation*

Ions carry electric current through a solution. How *well* a solution conducts depends on how many ions it contains and how fast they move. In this lab you will measure the **conductance** of four electrolytes, HCl, NaCl, sodium acetate (NaAc) and acetic acid (HAc), each at four concentrations. From those measurements you will:

1. calibrate your conductance cell with a KCl standard (the **cell constant** $k$),
2. turn raw conductances into **conductivities** $\kappa$ and **molar conductivities** $\Lambda_m$,
3. use **Kohlrausch's law** to find the limiting molar conductivity $\Lambda_m^\circ$ of the three strong electrolytes,
4. use the **law of independent migration of ions** to get $\Lambda_m^\circ$ for the weak acid HAc, which can't be found by extrapolation, and
5. find the **degree of ionization** $\alpha$ of acetic acid and, with a **Debye–Hückel** activity correction, its thermodynamic acidity constant $K_a$.

You don't need to know Python already. Each code cell explains what it does. 🐍 **Python tip** boxes introduce one idea at a time, and most answers are one line. Cells marked *"Just run this cell"* hold longer code that you don't need to read.

<br>**<center>Learning Goals**

|Area|Concept|
|---|---|
|Electrochemistry|Resistance, conductance, conductivity $\kappa$, cell constant $k = l/A$|
|Transport|Ionic mobility $U_i$; why H⁺ is so fast|
|Solution theory|Molar conductivity $\Lambda_m$; strong vs. weak electrolytes|
|Kohlrausch's law|$\Lambda_m = \Lambda_m^\circ - K\sqrt{c}$; extrapolating to infinite dilution|
|Independent migration|$\Lambda_m^\circ(\text{HAc}) = \Lambda_m^\circ(\text{HCl}) + \Lambda_m^\circ(\text{NaAc}) - \Lambda_m^\circ(\text{NaCl})$|
|Equilibrium|Degree of ionization $\alpha$; $K_c$ vs. $K_a$; Debye–Hückel activity coefficients|
|Data entry|Typing lab data into the notebook grid; saving it as a CSV file|
|Statistics|Mean and standard deviation of replicate runs; least-squares lines with standard errors|
|Uncertainty|Propagating uncertainty through a sum and through $10^x$|
|Python|Variables, functions, dictionaries, `for` loops, `numpy` arrays, `datascience` Tables, plotting|

## ⚙️ Initialization — DO NOT EDIT
The next two cells load the course tools (auto-checks, quizzes, styling) and record the lab version. **Do not change them** — with one exception:

👁️ **Accessible view (optional):** set `ACCESSIBLE_VIEW = True` in the cell below for larger, more readable text and colorblind-safe plots. You can switch back any time by setting it to `False` and re-running the cell (then *Kernel → Restart* to clear the text styling).

👉 **Run the cell below first** (Shift+Enter) before anything else in this lab.

In [ ]:
# 🔧 Run this cell first! (Shift+Enter)
ACCESSIBLE_VIEW = True     # True = larger text and colorblind-safe plots

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "shared-public")))
if ACCESSIBLE_VIEW:
    from init_access import *
else:
    from init import *

In [ ]:
# 🔖 Lab version — do not edit. Used by the automated checks below.
ver = '2026CondV1'

### 🧰 Standard imports
These lines load the tools used in this lab: `numpy` for math on arrays of numbers, `datascience` for tables, `matplotlib` for plots, `scipy.stats` for straight-line fits, and `pchem_entry` (in this lab's folder) for the data-entry grid.

In [ ]:
# Standard imports for this lab
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt
from scipy import stats
from ipywidgets import interact, FloatSlider, Checkbox
from pchem_entry import data_entry, load_entry, save_entry

Now run the cell below to enter your name.

In [ ]:
# Enter your name as a string, e.g. name = 'Svante Arrhenius'
name = ...

🐍 **Python tip — variables and units.** `F = 96485.33212` stores a number under the name `F`. Python doesn't track units, so **we put the units in a comment** (`# C mol⁻¹`) or in the name (`c_M` for mol L⁻¹, `G_uS` for microsiemens). Constants are below; the Faraday constant is from [CODATA 2018 (NIST)](https://physics.nist.gov/cuu/Constants/).

In [ ]:
# Constants used in this lab
F     = 96485.33212     # C mol⁻¹       Faraday constant (CODATA 2018)
A_DH  = 0.509           # kg^½ mol^-½   Debye–Hückel constant for water at 25 °C
c_std = 1.0             # mol L⁻¹       standard concentration c°

print(f"F = {F} C/mol;  Debye–Hückel A = {A_DH}")

## 1. Background: How Solutions Conduct Electricity

### 1.1 Resistance, conductance and conductivity
When a voltage $V$ is applied across two electrodes dipped in a solution, a current $I$ flows. Ohm's law defines the **resistance** $R = V/I$ (ohms, Ω). Its reciprocal is the **conductance**:

$$ G = \frac{1}{R} \qquad \text{units: siemens, } 1\ \text{S} = 1\ \Omega^{-1} $$

Conductance depends on the *shape* of the sample as well as on what it is made of. A sample twice as long conducts half as well; a sample with twice the cross-section conducts twice as well:

$$ G = \kappa\,\frac{A}{l} $$

The constant $\kappa$ (kappa) is the **conductivity**, a property of the solution alone, in S m⁻¹. ([Garland, Nibler & Shoemaker, *Experiments in Physical Chemistry*, 8th ed., Exp. 17](https://www.mheducation.com/highered/product/experiments-physical-chemistry-garland-nibler/M9780072828429.html); [Atkins, *Physical Chemistry*, 11th ed.](https://global.oup.com/academic/product/atkins-physical-chemistry-9780198769866), §16B)

### 1.2 The cell constant
Our conductance cell is a glass dip cell with two platinum electrodes. Their area $A$ and separation $l$ aren't known precisely, and the electric field isn't perfectly uniform anyway. So we lump the geometry into one number, the **cell constant**:

$$ k = \frac{l}{A} \qquad\Longrightarrow\qquad \kappa = k\,G = \frac{k}{R} $$

We find $k$ by measuring a solution whose conductivity is accurately known: **0.02000 M KCl**. At 25 °C its conductivity is $\kappa_\text{KCl} = 0.2767$ S m⁻¹ (2767 µS cm⁻¹). Then

$$ k = \frac{\kappa_\text{KCl}}{\bar G_\text{KCl}} $$

Conductivity rises by about **2 % per °C** (ions move faster in warmer, less viscous water), so we correct $\kappa_\text{KCl}$ to the temperature you measured ([US EPA Method 120.1](https://19january2021snapshot.epa.gov/sites/static/files/2015-08/documents/method_120-1_1982.pdf)):

$$ \kappa_\text{KCl}(T) \approx 0.2767\ \text{S m}^{-1}\,\bigl[1 + 0.0193\,(T - 25\ ^\circ\text{C})\bigr] $$

### 1.3 Correcting for the water
Even very pure water conducts a little, from H⁺ and OH⁻ (and dissolved CO₂). Ultrapure water (18.2 MΩ cm) has $\kappa = 5.5\times10^{-6}$ S m⁻¹. So we measure the water used to make the solutions and subtract its conductance $\bar G_w$:

$$ G_s = \bar G - \bar G_w \qquad\qquad \kappa = k\,G_s $$

| Water type | Resistivity (MΩ cm) | Conductivity (mS m⁻¹) |
|---|---|---|
| Tap water | ~0.002–0.02 | ~5–50 |
| Deionized (ASTM II) | > 5 | ≤ 0.2 |
| Ultrapure (ASTM I) | 18.2 | 0.055 |

*Values at 25 °C; from [A. Connelly, "Laboratory water"](https://andyjconnelly.wordpress.com/2017/02/20/laboratory-water/) and ASTM D1193.*

### <font color=blue> **Question 1.1.** </font>
Complete the unit-conversion functions. You'll use them throughout the lab.
- `molar_to_SI(c_M)`: mol L⁻¹ → mol m⁻³ (there are 1000 L in 1 m³)
- `Lambda_SI_to_cgs(L_SI)`: S m² mol⁻¹ → S cm² mol⁻¹ (1 m² = 10⁴ cm²)
- `resistivity_MOhm_cm(kappa_S_m)`: conductivity (S m⁻¹) → resistivity in MΩ cm. Resistivity is $\rho = 1/\kappa$ in Ω m; 1 Ω m = 100 Ω cm.

Check: ultrapure water, $\kappa = 5.5\times10^{-6}$ S m⁻¹, should give about 18.2 MΩ cm.

🐍 **Python tip:** `5.5e-6` means $5.5\times10^{-6}$. `**` is a power: `10**4` is 10 000.

In [ ]:
def molar_to_SI(c_M):
    """Concentration: mol L⁻¹ → mol m⁻³."""
    return ...

def Lambda_SI_to_cgs(L_SI):
    """Molar conductivity: S m² mol⁻¹ → S cm² mol⁻¹."""
    return ...

def resistivity_MOhm_cm(kappa_S_m):
    """Conductivity (S m⁻¹) → resistivity (MΩ cm)."""
    return ...

print(f"0.0200 M = {molar_to_SI(0.0200)} mol/m³")
print(f"0.01264 S m²/mol = {Lambda_SI_to_cgs(0.01264):.1f} S cm²/mol")
print(f"Ultrapure water: {resistivity_MOhm_cm(5.5e-6):.1f} MΩ cm")

In [ ]:
check('tests/q1.1.py')

### 1.4 Where conductivity comes from: ion mobility
Each ion drifts through the solution at a speed proportional to the electric field. The proportionality constant is the **mobility** $U_i$ (m² s⁻¹ V⁻¹). For a 1:1 electrolyte A⁺B⁻ that is ionized to a fraction $\alpha$ at concentration $c$ (mol m⁻³), each ion species has concentration $\alpha c$, and

$$ \kappa = \alpha\,c\,F\,(U_+ + U_-) $$

where $F$ is the Faraday constant, the charge on a mole of electrons. ([Garland, Nibler & Shoemaker, *Experiments in Physical Chemistry*, 8th ed., Exp. 17](https://www.mheducation.com/highered/product/experiments-physical-chemistry-garland-nibler/M9780072828429.html), pp. 235–245)

| Ion | $U$ at 25 °C ($10^{-8}$ m² s⁻¹ V⁻¹) | | Ion | $U$ at 25 °C ($10^{-8}$ m² s⁻¹ V⁻¹) |
|---|---|---|---|---|
| H⁺ | 36.23 | | OH⁻ | 20.64 |
| Na⁺ | 5.19 | | Cl⁻ | 7.91 |
| K⁺ | 7.62 | | CH₃COO⁻ | 4.24 |

*Mobilities from [Atkins, *Physical Chemistry*, 11th ed.](https://global.oup.com/academic/product/atkins-physical-chemistry-9780198769866), Table 16B.2 (CH₃COO⁻ calculated from its limiting ionic conductivity, CRC Handbook).*

H⁺ is about **7 times** faster than Na⁺. It doesn't have to push its way through the water at all. A proton hops from one water molecule to the next along hydrogen bonds, the **Grotthuss mechanism**. Small ions such as Li⁺ and Na⁺ hold a thick shell of water molecules, so their *hydrated* radius is large and they move slowly. ([Agmon, *Chem. Phys. Lett.* **244**, 456 (1995)](https://doi.org/10.1016/0009-2614(95)00905-J))

### <font color=blue> **Question 1.2.** </font>
Use the mobilities in the table to predict, for a fully ionized ($\alpha = 1$) **0.0100 M** solution:
- `kappa_NaCl` and `kappa_HCl`: the conductivity in S m⁻¹ (remember $c$ must be in **mol m⁻³**; use `molar_to_SI`)
- `Lambda_NaCl` and `Lambda_HCl`: the molar conductivity $\Lambda_m = \kappa/c$ in **S cm² mol⁻¹** (use `Lambda_SI_to_cgs`)

The mobilities are stored for you in a **dictionary**.

🐍 **Python tip — dictionaries.** `U = {'H': 36.23e-8, 'Na': 5.19e-8}` links each *key* (`'H'`) to a *value*. `U['Na']` gives `5.19e-08`.

In [ ]:
U = {'H': 36.23e-8, 'Na': 5.19e-8, 'K': 7.62e-8,
     'Cl': 7.91e-8, 'OH': 20.64e-8, 'Ac': 4.24e-8}      # m² s⁻¹ V⁻¹ at 25 °C

c_test = molar_to_SI(0.0100)        # mol m⁻³

kappa_NaCl = ...
kappa_HCl  = ...
Lambda_NaCl = ...
Lambda_HCl  = ...

print(f"NaCl: κ = {kappa_NaCl:.4f} S/m,  Λ = {Lambda_NaCl:.1f} S cm²/mol")
print(f"HCl:  κ = {kappa_HCl:.4f} S/m,  Λ = {Lambda_HCl:.1f} S cm²/mol")

In [ ]:
check('tests/q1.2.py')

### 1.5 Molar conductivity and Kohlrausch's law
Conductivity grows with concentration simply because there are more ions. To compare electrolytes fairly we divide by concentration, giving the **molar conductivity**:

$$ \Lambda_m = \frac{\kappa}{c} \qquad \text{units: S m}^2\text{ mol}^{-1}\ (\text{often quoted in S cm}^2\text{ mol}^{-1}) $$

If ions moved completely independently, $\Lambda_m$ would be constant. It isn't. Each ion is surrounded by an **ionic atmosphere** of opposite charge that drags on it, and the drag grows with concentration. In 1900 Friedrich Kohlrausch found that, for **strong** electrolytes in dilute solution,

$$ \boxed{\;\Lambda_m = \Lambda_m^\circ - K\sqrt{c/c^\circ}\;} $$

$\Lambda_m^\circ$ is the **limiting molar conductivity**, the value at infinite dilution where the ions no longer feel each other. $K$ is the Kohlrausch coefficient. A plot of $\Lambda_m$ against $\sqrt{c}$ is a straight line with intercept $\Lambda_m^\circ$. Debye, Hückel and Onsager later derived this law and predicted the slope for a 1:1 electrolyte in water at 25 °C ([Atkins, *Physical Chemistry*, 11th ed.](https://global.oup.com/academic/product/atkins-physical-chemistry-9780198769866), §16B.1):

$$ K = A' + B'\Lambda_m^\circ, \qquad A' = 60.20\ \text{S cm}^2\text{ mol}^{-1}\text{ (mol L}^{-1})^{-1/2},\quad B' = 0.2289\ (\text{mol L}^{-1})^{-1/2} $$

### 1.6 Weak electrolytes and independent migration
Acetic acid is only partly ionized, $\text{HAc} \rightleftharpoons \text{H}^+ + \text{Ac}^-$, and the fraction $\alpha$ that is ionized rises steeply as the solution is diluted. Its $\Lambda_m$ therefore shoots up near $c = 0$ and **can't be extrapolated** with a straight line. Instead we use Kohlrausch's **law of independent migration of ions**: at infinite dilution each ion contributes a fixed amount $\lambda^\circ$, whatever its partner.

$$ \Lambda_m^\circ(\text{HAc}) = \lambda^\circ_{\text{H}^+} + \lambda^\circ_{\text{Ac}^-} = \Lambda_m^\circ(\text{HCl}) + \Lambda_m^\circ(\text{NaAc}) - \Lambda_m^\circ(\text{NaCl}) $$

The Na⁺ and Cl⁻ contributions cancel.

### 1.7 From conductance to $K_a$
If the ions in a weak-acid solution are so dilute that they barely interact, their mobilities are the same as at infinite dilution, and

$$ \alpha \approx \frac{\Lambda_m}{\Lambda_m^\circ} $$

An ICE table for HAc at total concentration $c$ gives the **concentration** equilibrium constant:

| | HAc | H⁺ | Ac⁻ |
|---|---|---|---|
| Initial | $c$ | 0 | 0 |
| Change | $-\alpha c$ | $+\alpha c$ | $+\alpha c$ |
| Equilibrium | $c(1-\alpha)$ | $\alpha c$ | $\alpha c$ |

$$ K_c = \frac{[\text{H}^+][\text{Ac}^-]}{[\text{HAc}]} = \frac{\alpha^2 c}{1-\alpha} $$

The true thermodynamic constant uses **activities**, $a_i = \gamma_i[i]/c^\circ$. The neutral HAc has $\gamma \approx 1$, so $K_a = \gamma_\pm^2 K_c$, or $\log K_a = 2\log\gamma_\pm + \log K_c$. The **Debye–Hückel limiting law** gives the mean activity coefficient:

$$ \log\gamma_\pm = -|z_+z_-|\,\frac{A\sqrt{I}}{1+B\sqrt{I}} \approx -A\sqrt{\alpha c}, \qquad I = \tfrac12\sum_i c_i z_i^2 = \alpha c $$

with $A = 0.509$ kg$^{1/2}$ mol$^{-1/2}$ at 25 °C. Putting these together:

$$ \boxed{\;\log K_c = \log K_a + 2A\sqrt{\alpha c}\;} $$

Plot $\log K_c$ against $\sqrt{\alpha c}$: the **intercept is $\log K_a$** and the slope should be close to $2A = 1.018$. ([Garland, Nibler & Shoemaker, *Experiments in Physical Chemistry*, 8th ed., Exp. 17](https://www.mheducation.com/highered/product/experiments-physical-chemistry-garland-nibler/M9780072828429.html), pp. 235–245; [McQuarrie & Simon, *Physical Chemistry: A Molecular Approach*, Ch. 25](https://uscibooks.aip.org/books/physical-chemistry-a-molecular-approach/))

### 1.8 Strong vs. weak on a Kohlrausch plot
**Run the cell, then drag the slider.** The blue line is a strong 1:1 electrolyte that follows Kohlrausch's law. The orange curve is a weak acid with the same $\Lambda_m^\circ$ as HAc (390.6 S cm² mol⁻¹) and acidity constant $K_a$. Our measurement range, about 0.0003–0.02 M, is shaded. Try $\log_{10}K_a = -4.76$ (acetic acid), then make the acid stronger.

In [ ]:
# Just run this cell, then drag the slider.
def kohlrausch_explorer(log10_Ka=-4.76, show_strong=True):
    c = np.linspace(1e-7, 0.025, 2000)                        # mol L⁻¹
    L0_weak = 390.6                                           # S cm² mol⁻¹ (HAc, 25 °C)
    Ka = 10**log10_Ka
    alpha = (-Ka + np.sqrt(Ka**2 + 4 * Ka * c)) / (2 * c)    # from Ka = α²c/(1-α), activities ignored
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.axvspan(np.sqrt(0.02 / 64), np.sqrt(0.02), color='gray', alpha=0.12, label='range measured in this lab')
    if show_strong:
        ax.plot(np.sqrt(c), 126.4 - 89.1 * np.sqrt(c), lw=2.5, label=r'strong (NaCl): $\Lambda^\circ - K\sqrt{c}$')
    ax.plot(np.sqrt(c), alpha * L0_weak, lw=2.5, label=fr'weak acid, $K_a$ = {Ka:.2e}')
    ax.scatter([0], [L0_weak], s=80, color='C1', zorder=4)
    ax.text(0.004, L0_weak - 25, r'$\Lambda^\circ$(HAc)', color='C1')
    ax.set_xlim(0, 0.16); ax.set_ylim(0, 420)
    ax.set_xlabel(r'$\sqrt{c}$  (mol L$^{-1}$)$^{1/2}$')
    ax.set_ylabel(r'$\Lambda_m$  (S cm$^2$ mol$^{-1}$)')
    ax.set_title('Why a weak electrolyte cannot be extrapolated to $c = 0$')
    ax.legend(loc='upper right')
    plt.show()

interact(kohlrausch_explorer,
         log10_Ka=FloatSlider(min=-7, max=-1, step=0.1, value=-4.76, description='log₁₀ Kₐ'),
         show_strong=Checkbox(value=True, description='show NaCl'));

### <font color=blue> **Question 1.3 Discussion.** </font> 
<font color=green>
1. Your answer to Question 1.2 gives $\Lambda_m$(HCl) about 3.5 times $\Lambda_m$(NaCl), even though both contain Cl⁻. Explain why, using the mobilities and the Grotthuss mechanism.
2. Na⁺ is a *smaller* ion than K⁺, yet it moves more slowly. Why?
3. Using the slider: in the shaded concentration range, why would a straight-line fit to the weak-acid curve give a badly wrong $\Lambda_m^\circ$? What happens to the curve as $K_a$ gets larger, and why?
</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q1.3d.py')

## 2. Solution Concentrations

Each electrolyte is made up as a stock solution of concentration $c$, then diluted to $c/4$, $c/16$ and $c/64$ (each a 1 → 4 dilution of the one before).

The HCl stock was made from concentrated Fisher HCl. Its label reads:

> *Hydrochloric acid (HCl) 12.1 N — 5.10 mL ± 0.05 mL added in 3 L (total volume) 18 MΩ cm DI H₂O.*

For HCl, normality equals molarity (one H⁺ per formula unit). Since moles of HCl are conserved on dilution, $c_1V_1 = c_2V_2$.

### <font color=blue> **Question 2.1.** </font>
1. Compute `c_HCl`, the concentration of the HCl stock in mol L⁻¹ (keep both volumes in the same unit).
2. Fill in the stock concentrations of your other three solutions in the dictionary `c_stock`, from the labels on the bottles (or from your own mass and volume if you made them). The example values are 0.02000 M.

🐍 **Python tip:** a dictionary value can be a variable or a calculation: `{'HCl': c_HCl, 'NaCl': 0.02000}`.

In [ ]:
c_HCl = ...                    # mol L⁻¹

c_stock = {'HCl':  c_HCl,
           'NaCl': 0.02000,    # mol L⁻¹  ← use the value on YOUR bottle
           'NaAc': 0.02000,    # mol L⁻¹  ← use the value on YOUR bottle
           'HAc':  0.02000}    # mol L⁻¹  ← use the value on YOUR bottle

print(f"HCl stock: {c_HCl:.5f} M")

In [ ]:
check('tests/q2.1.py')

### <font color=blue> **Question 2.2.** </font>
Complete `concentration(solution, dilution)` so that it returns the molar concentration of any of your solutions, for example `concentration('HCl', 'c/16')`. Use the two dictionaries `c_stock` and `dilution_factor`.

🐍 **Python tip — functions.** `def name(inputs):` defines a reusable recipe and `return` hands back the answer.

In [ ]:
dilution_factor = {'c': 1, 'c/4': 4, 'c/16': 16, 'c/64': 64}

def concentration(solution, dilution):
    """Molar concentration (mol L⁻¹) of one solution, e.g. concentration('HAc', 'c/4')."""
    return ...

for d in ['c', 'c/4', 'c/16', 'c/64']:
    print(f"HCl {d:>4}: {concentration('HCl', d):.6f} M")

In [ ]:
check('tests/q2.2.py')

## 3. Enter Your Data

The grid below has the same layout as your paper data sheet: one row per solution, four replicate conductance readings (**Run 1–4**) and the temperature for each run. The labels are already filled in. **It saves automatically** to `conductance_data.csv` every time you leave a box, so your data survive a kernel restart. If you run the cell again, it reloads what you saved.

| Column | What to enter |
|---|---|
| `Set` | `A` for the HAc / NaAc session, `B` for the NaCl / HCl session. Each set has its own water and KCl readings, so each gets its own cell constant. |
| `Solution` | `Water`, `KCl`, `HCl`, `NaCl`, `NaAc` or `HAc` (spell exactly like this) |
| `Dilution` | `c`, `c/4`, `c/16`, `c/64`, or `-` for water and KCl |
| `G1_uS` … `G4_uS` | raw conductance, Runs 1–4, in **µS** (microsiemens) |
| `T1_C` … `T4_C` | temperature for Runs 1–4, °C |

- ⚠️ **Units:** if your meter reads in **mS**, multiply by 1000 before typing, or change `G_UNIT` in Section 3.1.
- Leave a box blank if you skipped a run; blanks are ignored in the averages.
- No data yet? Click **📋 Load example data** to practice. **Replace it with your own data before you submit.**
- A box outlined in orange holds something that isn't a number.

*Example data are **simulated** for a cell with $k \approx 99$ m⁻¹ at about 23 °C, using literature limiting ionic conductivities, the Onsager equation and $K_a$ = 1.75 × 10⁻⁵, with 0.2 % random noise. They are for practice only.*

In [ ]:
# Run this cell to open the data-entry grid.
DATA_FILE = 'conductance_data.csv'
columns = ['Set', 'Solution', 'Dilution',
           'G1_uS', 'G2_uS', 'G3_uS', 'G4_uS', 'T1_C', 'T2_C', 'T3_C', 'T4_C']
labels  = {'Set': 'Set', 'Solution': 'Solution', 'Dilution': 'Dilution',
           'G1_uS': 'G Run 1 (µS)', 'G2_uS': 'G Run 2 (µS)', 'G3_uS': 'G Run 3 (µS)', 'G4_uS': 'G Run 4 (µS)',
           'T1_C': 'T 1 (°C)', 'T2_C': 'T 2 (°C)', 'T3_C': 'T 3 (°C)', 'T4_C': 'T 4 (°C)'}
widths  = {'Set': '45px', 'Solution': '75px', 'Dilution': '65px'}

data_entry(columns, DATA_FILE, n_rows=20, labels=labels, widths=widths, box_width='90px',
           text_columns=['Set', 'Solution', 'Dilution'],
           template='conductance_template.csv', example='example_conductance_data.csv')

<details><summary>⌨️ <b>Prefer typing in code?</b> (click to expand)</summary>

Instead of the grid, you can put your data in a code cell like this and run it. It writes the same `conductance_data.csv` file (add one entry per row to every list):

```python
save_entry({'Set':      ['A', 'A', 'A'],
            'Solution': ['Water', 'KCl', 'HAc'],
            'Dilution': ['-', '-', 'c/64'],
            'G1_uS': [1.19, 2713, 26.7], 'G2_uS': [1.19, 2709, 26.7],
            'G3_uS': [1.15, 2708, 26.7], 'G4_uS': [1.24, 2710, 26.7],
            'T1_C': [23.3, 23.2, 23.2],  'T2_C': [23.1, 23.2, 23.3],
            'T3_C': [23.2, 23.2, 23.2],  'T4_C': [23.2, 23.3, 23.1]}, DATA_FILE)
```
</details>

Now load the saved file into a **Table**. The grid itself won't show in your PDF, so this printed table is the record of your data.

🐍 **Python tip — Tables.** A `datascience` Table is like a spreadsheet. `tbl.column('G1_uS')` gives one column as a `numpy` **array**. `tbl.where('Solution', 'KCl')` keeps only the KCl rows.

In [ ]:
cond = load_entry(DATA_FILE, text_columns=['Set', 'Solution', 'Dilution'])
cond.show()

### <font color=blue> **Question 3.1.** </font>
Check that your data loaded. The test passes when `cond` contains Water and KCl rows plus at least three concentrations of each of HCl, NaCl, NaAc and HAc.

In [ ]:
check('tests/q3.1.py')

### 3.1 Average the replicate runs
### <font color=blue> **Question 3.2.** </font>
The cell below stacks the four runs side by side into a 2-D array `G_runs` (one row per solution, one column per run). Compute, for every row,
- `G_mean`: the mean of the runs
- `G_sd`: the sample standard deviation (use `ddof=1`)
- `T_mean`: the mean temperature

🐍 **Python tip — averaging along an axis.** For a 2-D array, `np.nanmean(G_runs, axis=1)` averages *across each row* and ignores blank (`nan`) entries. `np.nanstd(G_runs, axis=1, ddof=1)` does the same for the standard deviation. `ddof=1` divides by $n-1$, the sample standard deviation.

In [ ]:
G_UNIT = 1e-6        # your readings are in µS → multiply by 1e-6 to get S  (use 1e-3 if you typed mS)

G_runs = np.column_stack([cond.column(f'G{i}_uS') for i in (1, 2, 3, 4)])
T_runs = np.column_stack([cond.column(f'T{i}_C') for i in (1, 2, 3, 4)])

G_mean = ...
G_sd   = ...
T_mean = ...

summary = Table().with_columns(
    'Set', cond.column('Set'), 'Solution', cond.column('Solution'), 'Dilution', cond.column('Dilution'),
    'n runs', np.sum(~np.isnan(G_runs), axis=1),
    'G_mean (µS)', np.round(G_mean, 3), 'G_sd (µS)', np.round(G_sd, 3), 'T_mean (°C)', np.round(T_mean, 2))
summary.show()

In [ ]:
check('tests/q3.2.py')

## 4. The Cell Constant

### <font color=blue> **Question 4.1.** </font>
1. Complete `kappa_KCl(T_C)`, the conductivity of 0.02000 M KCl in S m⁻¹ at temperature `T_C` (°C), using the equation in §1.2.
2. In the loop, compute the cell constant for each set: $k = \kappa_\text{KCl}(T)/\bar G_\text{KCl}$ (in m⁻¹, with $\bar G$ in S).

🐍 **Python tip — `for` loops.** `for s in sets:` runs the indented lines once for each set (`'A'`, then `'B'`), with `s` taking each value in turn. `k_cell[s] = ...` stores a result in the dictionary under that set's name.

In [ ]:
def kappa_KCl(T_C):
    """Conductivity (S m⁻¹) of 0.02000 M KCl at T_C (°C)."""
    return ...

def lookup(set_name, solution, column):
    """One value from `summary` for a given set and solution (water or KCl)."""
    rows = summary.where('Set', set_name).where('Solution', solution)
    return rows.column(column).item(0)

sets = np.unique(summary.column('Set'))
Gw_bar = {}          # S     mean conductance of the water, per set
k_cell = {}          # m⁻¹   cell constant, per set
for s in sets:
    Gw_bar[s] = lookup(s, 'Water', 'G_mean (µS)') * G_UNIT
    G_KCl     = lookup(s, 'KCl',   'G_mean (µS)') * G_UNIT
    T_KCl     = lookup(s, 'KCl',   'T_mean (°C)')
    k_cell[s] = ...
    print(f"Set {s}: T = {T_KCl:.2f} °C, κ_KCl = {kappa_KCl(T_KCl):.4f} S/m, "
          f"G_KCl = {G_KCl*1e3:.4f} mS, G_water = {Gw_bar[s]*1e6:.2f} µS  →  k = {k_cell[s]:.2f} m⁻¹")

In [ ]:
check('tests/q4.1.py')

Run the cell below to check the purity of your water: it converts $k\bar G_w$ into a resistivity to compare with the water-quality table in §1.3.

In [ ]:
# Just run this cell
for s in sets:
    kappa_w = k_cell[s] * Gw_bar[s]
    print(f"Set {s}: water κ = {kappa_w*1e3:.4f} mS/m  →  resistivity {resistivity_MOhm_cm(kappa_w):.3f} MΩ cm")

## 5. Conductivity and Molar Conductivity

### <font color=blue> **Question 5.1.** </font>
The first lines pick out the electrolyte rows and line up, row by row, the concentration and the cell constant and water conductance for that row's set. Complete the three equations (all SI units):

$$ G_s = \bar G - \bar G_w \qquad \kappa = k\,G_s \qquad \Lambda_m = \frac{\kappa}{c} $$

with $c$ in **mol m⁻³** so that $\Lambda_m$ comes out in S m² mol⁻¹. The last line converts to S cm² mol⁻¹, the unit used in most tables.

🐍 **Python tip — list comprehensions.** `[concentration(s, d) for s, d in zip(sol, dil)]` walks through two lists together and makes a new list with one result per pair. Math on `numpy` arrays then works on every row at once.

In [ ]:
samples = summary.where('Dilution', are.not_equal_to('-'))
sol, dil, st = samples.column('Solution'), samples.column('Dilution'), samples.column('Set')

c_M   = np.array([concentration(s, d) for s, d in zip(sol, dil)])   # mol L⁻¹
k_row = np.array([k_cell[s] for s in st])                           # m⁻¹
Gw_row = np.array([Gw_bar[s] for s in st])                          # S
G_bar = samples.column('G_mean (µS)') * G_UNIT                      # S

G_s      = ...
kappa    = ...
Lambda_m = ...                                                      # S m² mol⁻¹
Lambda   = Lambda_SI_to_cgs(Lambda_m)                               # S cm² mol⁻¹

In [ ]:
check('tests/q5.1.py')

Run the cell below to collect everything in one results table and plot $\Lambda_m$ against $\sqrt{c}$ for all four electrolytes. The error bars are ± one standard deviation of the four runs, carried through to $\Lambda_m$ ($\sigma_\Lambda = k\,\sigma_G/c$).

In [ ]:
# Just run this cell: results table and Kohlrausch plot
Lambda_sd = Lambda_SI_to_cgs(k_row * samples.column('G_sd (µS)') * G_UNIT / molar_to_SI(c_M))
results = samples.select('Set', 'Solution', 'Dilution', 'T_mean (°C)').with_columns(
    'c (M)', c_M, 'sqrt c', np.sqrt(c_M), 'kappa (S/m)', kappa,
    'Lambda (S cm2/mol)', Lambda, 'Lambda_sd', Lambda_sd)
results.set_format(['c (M)', 'kappa (S/m)'], NumberFormatter(6))
results.set_format(['sqrt c', 'Lambda (S cm2/mol)', 'Lambda_sd'], NumberFormatter(3))
results.show()

electrolytes = ['HCl', 'NaCl', 'NaAc', 'HAc']
markers = {'HCl': 'o', 'NaCl': 's', 'NaAc': '^', 'HAc': 'D'}
fig, ax = plt.subplots(figsize=(8, 5.5))
for i, e in enumerate(electrolytes):
    sub = results.where('Solution', e)
    ax.errorbar(sub.column('sqrt c'), sub.column('Lambda (S cm2/mol)'), yerr=sub.column('Lambda_sd'),
                marker=markers[e], ms=8, ls='-', lw=1, color=f'C{i}', capsize=3, label=e)
ax.set_xlim(left=0); ax.set_ylim(bottom=0)
ax.set_xlabel(r'$\sqrt{c}$  (mol L$^{-1}$)$^{1/2}$')
ax.set_ylabel(r'$\Lambda_m$  (S cm$^2$ mol$^{-1}$)')
ax.set_title('Molar conductivity vs. square root of concentration')
ax.legend()
plt.show()

## 6. Kohlrausch's Law: Limiting Molar Conductivities

For the three **strong** electrolytes, fit a straight line $\Lambda_m = \Lambda_m^\circ - K\sqrt{c}$. The intercept is $\Lambda_m^\circ$ and the slope is $-K$.

The helper `linear_fit` uses least squares (`scipy.stats.linregress`). Besides the slope and intercept, it returns their **standard errors**, the uncertainties that come from how much the points scatter about the line ([Taylor, *An Introduction to Error Analysis*, Ch. 8](https://uscibooks.aip.org/books/introduction-to-error-analysis-2nd-ed/)).

### <font color=blue> **Question 6.1.** </font>
Complete the body of the loop so that, for each strong electrolyte `e`, it stores
- `Lambda0[e]`: the intercept $\Lambda_m^\circ$ (S cm² mol⁻¹)
- `Lambda0_err[e]`: its standard error
- `K_kohl[e]`: the Kohlrausch coefficient $K$ (a **positive** number)

🐍 **Python tip:** a function can return several values at once. `slope, intercept, slope_err, intercept_err = linear_fit(x, y)` unpacks them into four names.

In [ ]:
def linear_fit(x, y):
    """Least-squares line y = slope*x + intercept. Returns slope, intercept and their standard errors."""
    fit = stats.linregress(x, y)
    return fit.slope, fit.intercept, fit.stderr, fit.intercept_stderr

strong = ['HCl', 'NaCl', 'NaAc']
Lambda0, Lambda0_err, K_kohl = {}, {}, {}
for e in strong:
    sub = results.where('Solution', e)
    slope, intercept, slope_err, intercept_err = linear_fit(sub.column('sqrt c'), sub.column('Lambda (S cm2/mol)'))
    Lambda0[e]     = ...
    Lambda0_err[e] = ...
    K_kohl[e]      = ...
    print(f"{e:5s}: Λ° = {Lambda0[e]:6.1f} ± {Lambda0_err[e]:.1f} S cm²/mol,   K = {K_kohl[e]:6.1f}")

In [ ]:
check('tests/q6.1.py')

Run the cell below to see the fits and compare with literature values at 25 °C and with the Debye–Hückel–Onsager prediction $K = A' + B'\Lambda_m^\circ$.

In [ ]:
# Just run this cell: Kohlrausch fits and comparison with literature
Lambda0_lit = {'HCl': 425.96, 'NaCl': 126.39, 'NaAc': 90.98, 'HAc': 390.55}   # S cm² mol⁻¹, 25 °C (CRC ionic values)

fig, ax = plt.subplots(figsize=(8, 5.5))
xx = np.linspace(0, 0.16, 50)
for i, e in enumerate(strong):
    sub = results.where('Solution', e)
    ax.errorbar(sub.column('sqrt c'), sub.column('Lambda (S cm2/mol)'), yerr=sub.column('Lambda_sd'),
                marker=markers[e], ms=8, ls='none', color=f'C{i}', capsize=3, label=f'{e} data')
    ax.plot(xx, Lambda0[e] - K_kohl[e] * xx, color=f'C{i}', lw=1.5, ls='--')
    ax.scatter([0], [Lambda0_lit[e]], marker='*', s=180, color=f'C{i}', edgecolor='black', zorder=5,
               label=f'{e} literature Λ° (25 °C)')
ax.set_xlim(left=-0.003); ax.set_ylim(0, 470)
ax.set_xlabel(r'$\sqrt{c}$  (mol L$^{-1}$)$^{1/2}$')
ax.set_ylabel(r'$\Lambda_m$  (S cm$^2$ mol$^{-1}$)')
ax.set_title("Kohlrausch extrapolation for the strong electrolytes")
ax.legend(fontsize=9, ncol=2, loc='center right')
plt.show()

comparison = Table().with_columns(
    'Electrolyte', strong,
    'Λ° this lab', [round(Lambda0[e], 1) for e in strong],
    '± (SE)', [round(Lambda0_err[e], 1) for e in strong],
    'Λ° lit. 25 °C', [Lambda0_lit[e] for e in strong],
    '% diff', [round(100 * (Lambda0[e] - Lambda0_lit[e]) / Lambda0_lit[e], 1) for e in strong],
    'K this lab', [round(K_kohl[e], 1) for e in strong],
    'K Onsager', [round(60.20 + 0.2289 * Lambda0[e], 1) for e in strong])
comparison.show()

### <font color=blue> **Question 6.2 Discussion.** </font> 
<font color=green>
1. How do your $\Lambda_m^\circ$ values compare with the literature values at 25 °C? Look at the temperatures in your data. Conductivities rise about 2 % per °C, so is the size **and sign** of your difference what you'd expect?
2. Compare your Kohlrausch coefficients $K$ with the Debye–Hückel–Onsager prediction. Why does HCl have the largest $K$? (Look at the formula $K = A' + B'\Lambda_m^\circ$.)
3. Kohlrausch's law is a *limiting* law for dilute solutions. Do your highest-concentration points (the stock, $c$) fall on the line? If not, would leaving them out raise or lower $\Lambda_m^\circ$?
</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q6.2d.py')

## 7. Acetic Acid: Degree of Ionization and $K_a$

### <font color=blue> **Question 7.1.** </font>
Use the law of independent migration (§1.6) to compute `Lambda0_HAc` from your three fitted $\Lambda_m^\circ$ values.

For a sum or difference, the **absolute** uncertainties add in quadrature:

$$ \sigma_{\Lambda^\circ(\text{HAc})} = \sqrt{\sigma^2_\text{HCl} + \sigma^2_\text{NaAc} + \sigma^2_\text{NaCl}} $$

Store it in `Lambda0_HAc_err`. Use `np.sqrt`.

In [ ]:
Lambda0_HAc = ...
Lambda0_HAc_err = ...

print(f"Λ°(HAc) = {Lambda0_HAc:.1f} ± {Lambda0_HAc_err:.1f} S cm²/mol   (literature, 25 °C: {Lambda0_lit['HAc']})")

In [ ]:
check('tests/q7.1.py')

### <font color=blue> **Question 7.2.** </font>
For each HAc concentration compute (as arrays):
- `alpha`: the degree of ionization, $\alpha = \Lambda_m/\Lambda_m^\circ$
- `Kc`: the concentration equilibrium constant, $K_c = \alpha^2 c/(1-\alpha)$, with $c$ in mol L⁻¹
- `x_DH`: the Debye–Hückel variable $\sqrt{\alpha c}$
- `log_Kc`: $\log_{10}K_c$ (use `np.log10`)

In [ ]:
hac = results.where('Solution', 'HAc').sort('c (M)')
c_HAc = hac.column('c (M)')                     # mol L⁻¹
Lambda_HAc = hac.column('Lambda (S cm2/mol)')   # S cm² mol⁻¹

alpha  = ...
Kc     = ...
x_DH   = ...
log_Kc = ...

hac_table = hac.select('Dilution', 'c (M)', 'Lambda (S cm2/mol)').with_columns(
    'alpha', alpha, 'Kc', Kc, 'sqrt(alpha c)', x_DH, 'log Kc', log_Kc)
hac_table.set_format(['alpha', 'sqrt(alpha c)', 'log Kc'], NumberFormatter(4))
hac_table.set_format('Kc', lambda v: f"{v:.3e}")
hac_table.show()

In [ ]:
check('tests/q7.2.py')

### <font color=blue> **Question 7.3.** </font>
Fit a straight line to $\log K_c$ versus $\sqrt{\alpha c}$ using `linear_fit`. Then compute
- `log_Ka` and `log_Ka_err`: the intercept and its standard error
- `Ka`: $10^{\log K_a}$, and `pKa` $= -\log K_a$
- `Ka_err`: for $K_a = 10^{y}$, $\;\sigma_{K_a} = K_a \ln(10)\,\sigma_y$ (use `np.log(10)`)
- `slope_DH`: the fitted slope, to compare with $2A = 1.018$

In [ ]:
slope, intercept, slope_err, intercept_err = linear_fit(x_DH, log_Kc)

log_Ka     = ...
log_Ka_err = ...
Ka         = ...
pKa        = ...
Ka_err     = ...
slope_DH   = ...

print(f"log Ka = {log_Ka:.3f} ± {log_Ka_err:.3f}")
print(f"Ka = ({Ka*1e5:.2f} ± {Ka_err*1e5:.2f}) × 10⁻⁵,   pKa = {pKa:.3f}")
print(f"slope = {slope_DH:.2f} ± {slope_err:.2f}   (Debye–Hückel: 2A = {2*A_DH:.3f})")

In [ ]:
check('tests/q7.3.py')

Run the cell below to plot the fit. It also shows a second estimate: **fix the slope at the theoretical $2A$** and average $\log K_c - 2A\sqrt{\alpha c}$ over your points. With only four points, fixing the slope often gives a more stable intercept.

In [ ]:
# Just run this cell: log Kc vs sqrt(alpha c)
Ka_lit = 1.754e-5          # acetic acid, 25 °C (Harned & Ehlers 1933)
log_Ka_fixed = np.mean(log_Kc - 2 * A_DH * x_DH)

xx = np.linspace(0, 1.1 * np.max(x_DH), 50)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_DH, log_Kc, 'o', ms=9, color='C3', label='HAc data')
ax.plot(xx, log_Ka + slope_DH * xx, color='C3', lw=1.5, label=f'free fit: slope {slope_DH:.2f}, Ka = {Ka:.2e}')
ax.plot(xx, log_Ka_fixed + 2 * A_DH * xx, color='C0', lw=1.5, ls='--',
        label=f'slope fixed at 2A = {2*A_DH:.3f}, Ka = {10**log_Ka_fixed:.2e}')
ax.axhline(np.log10(Ka_lit), color='gray', lw=1, ls=':')
ax.text(0.3 * xx[-1], np.log10(Ka_lit), f' literature log Ka = {np.log10(Ka_lit):.3f}', color='gray', fontsize=10, va='bottom')
ax.set_xlabel(r'$\sqrt{\alpha c}$  (mol L$^{-1}$)$^{1/2}$')
ax.set_ylabel(r'$\log_{10} K_c$')
ax.set_title(r'Thermodynamic $K_a$ from the intercept')
ax.legend(fontsize=9, loc='lower right')
plt.show()

print(f"Free fit:    Ka = {Ka:.3e} ± {Ka_err:.1e}   ({100*(Ka-Ka_lit)/Ka_lit:+.1f} % from literature)")
print(f"Fixed slope: Ka = {10**log_Ka_fixed:.3e}            ({100*(10**log_Ka_fixed-Ka_lit)/Ka_lit:+.1f} % from literature)")
print(f"Literature:  Ka = {Ka_lit:.3e},  pKa = {-np.log10(Ka_lit):.3f}")

### <font color=blue> **Question 7.4 Discussion.** </font> 
<font color=green>
1. Report your $K_a$ and p$K_a$ with uncertainty, and compare them with the literature value ($K_a = 1.754\times10^{-5}$, p$K_a$ = 4.756 at 25 °C). Which estimate is closer, the free fit or the fixed slope? Why might fixing the slope help with only four points?
2. $K_a$ for acetic acid hardly changes between 20 and 25 °C, yet your $\Lambda_m^\circ$ values in Section 6 were affected by temperature. Why does the temperature effect largely **cancel** in $\alpha = \Lambda_m/\Lambda_m^\circ$? (Hint: both were measured near the same temperature.)
3. The approximation $\alpha \approx \Lambda_m/\Lambda_m^\circ$ assumes the ions in the HAc solution move as fast as at infinite dilution. In reality the ionic atmosphere slows them a little. Does that make your $\alpha$, and hence $K_a$, too high or too low?
4. At $c/64$ the HAc conductance is only about 20 times the water conductance. What happens to $K_c$ at that concentration if $\bar G_w$ is off by 0.5 µS? Which data points does the water correction matter most for?
5. **Open question.** Your strong-electrolyte data were measured in Set B and the HAc data in Set A. What assumptions does combining them require, and how could you design the experiment to make $K_a$ more accurate?
</font>

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q7.4d.py')

### 📝 Quick Check — Conductance and Weak Electrolytes
Test what you've learned. This is just for practice — answer as many times as you like, it isn't recorded.

In [ ]:
with open("questionsCondA.json", "r") as f:
    display_quiz(json.load(f), shuffle_answers=True)

## 8. Reflection

### <font color=blue> **Question 8.** </font>
At the end of each lab, please include a reflection.
* How did this lab go? Was entering your data in the notebook grid straightforward?
* Which part of the Python was new to you, and which part was most useful?
* Did you try the accessible view? If so, was it helpful?
* Were there questions you found especially challenging that you would like your instructor to review in class?
* How long did the lab take you to complete?

**Use the markdown cell below this one and write your reflection on this lab.**

---

ANSWER

---

#### Save notebook before check

In [ ]:
check('tests/q8d.py')

## 📚 References
1. C. W. Garland, J. W. Nibler and D. P. Shoemaker, *Experiments in Physical Chemistry*, 8th ed., McGraw-Hill (2009), Experiment 17, "Conductance of Solutions," pp. 235–245.
2. P. Atkins and J. de Paula, *Atkins' Physical Chemistry*, 11th ed., Oxford University Press (2018), §5F (Debye–Hückel theory) and §16B (ion mobility, Kohlrausch's law, molar conductivity).
3. D. A. McQuarrie and J. D. Simon, *Physical Chemistry: A Molecular Approach*, University Science Books (1997), Ch. 25 (solutions of electrolytes).
4. A. Hamnett, "Thermodynamics of electrolyte solutions," in *Handbook of Fuel Cells*, Vol. 1, Ch. 2, John Wiley, Chichester (2003), pp. 20–24 — conductivity and equivalent-conductivity plots used in lecture.
5. F. Kohlrausch, *Das Leitvermögen der Elektrolyte, insbesondere der Lösungen*, Teubner, Leipzig (1898).
6. L. Onsager, "Zur Theorie der Elektrolyte. II," *Phys. Z.* **28**, 277–298 (1927).
7. H. S. Harned and R. W. Ehlers, "The dissociation constant of acetic acid from 0 to 60° centigrade," *J. Am. Chem. Soc.* **55**, 652–656 (1933), [doi:10.1021/ja01329a027](https://doi.org/10.1021/ja01329a027).
8. D. A. MacInnes and T. Shedlovsky, "The determination of the ionization constant of acetic acid, at 25°, from conductance measurements," *J. Am. Chem. Soc.* **54**, 1429–1438 (1932), [doi:10.1021/ja01343a020](https://doi.org/10.1021/ja01343a020).
9. N. Agmon, "The Grotthuss mechanism," *Chem. Phys. Lett.* **244**, 456–462 (1995), [doi:10.1016/0009-2614(95)00905-J](https://doi.org/10.1016/0009-2614(95)00905-J).
10. *CRC Handbook of Chemistry and Physics*, "Equivalent Conductivity of Electrolytes in Aqueous Solution" and "Ionic Conductivity and Diffusion at Infinite Dilution."
11. US EPA, Method 120.1, "Conductance (Specific Conductance, µmhos at 25 °C)" (1982), [PDF](https://19january2021snapshot.epa.gov/sites/static/files/2015-08/documents/method_120-1_1982.pdf) — KCl standard and temperature correction.
12. A. Connelly, "Laboratory water," [andyjconnelly.wordpress.com](https://andyjconnelly.wordpress.com/2017/02/20/laboratory-water/) (2017); ASTM D1193, *Standard Specification for Reagent Water*.
13. J. R. Taylor, *An Introduction to Error Analysis*, 2nd ed., University Science Books (1997).
14. E. Tiesinga et al., "CODATA Recommended Values of the Fundamental Physical Constants: 2018," *Rev. Mod. Phys.* **93**, 025010 (2021).

## 🏁 Final Step — Check, Then Submit to Canvas

Follow these steps **in order**:

1. ✅ **Run the cell below** to test all your answers and see your score.
2. 💾 **Save** the notebook (Ctrl/Cmd + S).
3. 📤 **Make a PDF and upload it to Canvas.** Export an HTML copy ([instructions](https://laserchemist.github.io/data/submission-instructions/export-html)), open it in your browser, and use **Print → Save as PDF**. Check that your data table, plots and written answers all appear in the PDF, then upload it to this lab's Canvas assignment.

In [ ]:
# Run all tests at once
from gofer.ok import check
correct = 0
questions = ['1.1', '1.2', '1.3d',
             '2.1', '2.2',
             '3.1', '3.2',
             '4.1',
             '5.1',
             '6.1', '6.2d',
             '7.1', '7.2', '7.3', '7.4d',
             '8d']
for x in questions:
    print("Testing question {}: ".format(x))
    display(check("tests/q{}.py".format(x)))
    score = check('tests/q{}.py'.format(str(x)))
    if score.grade == 1.0:
        correct += 1
print(f"{correct} of {len(questions)} checks passed")